# Fraud Pattern Deep-Dive

Analysis of monthly fraud rates, device/residence risk factors, and heuristic rule evaluation on cleaned data.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append("..")
from src.visualization.visualize import plot_fraud_by_category, plot_monthly_trend

df = pd.read_parquet("../data/processed/clean_base.parquet")


### 1. Monthly trend

In [ ]:
df.groupby("month").agg(
    total_apps=("fraud_bool", "count"),
    fraud_cases=("fraud_bool", "sum"),
    fraud_rate=("fraud_bool", lambda x: x.mean() * 100)
).reset_index()

### 2. Device OS & housing risk breakdown

In [ ]:
plot_fraud_by_category(df, "device_os", "Fraud Rate by Device OS")
plot_fraud_by_category(df, "housing_status", "Fraud Rate by Housing Status")

### 3. Heuristic rules evaluation
Compare precision and volume of simple business rules against 1.10% baseline.

In [ ]:
rule1 = (df["device_os"] == "windows") & (df["housing_status"] == "BA") & (df["has_prev_address"] == 0)
rule2 = (df["name_email_similarity"] < 0.3) & (df["email_is_free"] == 1) & (df["foreign_request"] == 1)
rule3 = (df["has_prior_bank_account"] == 0) & (df["no_valid_phone"] == 1)

eval_list = []
for name, mask in [("Windows + Housing BA + No Prev Addr", rule1),
                   ("Low Similarity + Free Email + Foreign", rule2),
                   ("No Prior Bank + No Valid Phone", rule3)]:
    triggered = mask.sum()
    caught = df.loc[mask, "fraud_bool"].sum()
    precision = (caught / triggered * 100) if triggered > 0 else 0
    eval_list.append({
        "Rule": name,
        "Triggered": int(triggered),
        "Frauds Caught": int(caught),
        "Precision (%)": round(precision, 2)
    })

pd.DataFrame(eval_list)